# Tratamento dos Dados — Mamografias


---
## 1. Imports

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

---
## 2. Configurações Gerais


In [2]:
# Caminhos
ENTRADA = Path("/home/anna/tcc/Limpeza e Integração/Mamografia/Dados")
SAIDA = Path("/home/anna/tcc/Seleção e Transformação/Mamografias/tabela_mamografias_por_faixa_etária.csv")
MUNICIPIOS = Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv")

# Faixas padronizadas e padrões de texto das colunas de origem
FAIXAS: list[tuple[str, list[str]]] = [
    ("0-9",       ["ate 9"]),
    ("10-19",     ["entre 10 a 14", "entre 15 a 19"]),
    ("20-29",     ["entre 20 a 24", "entre 25 a 29"]),
    ("30-39",     ["entre 30 a 34", "entre 35 a 39"]),
    ("40-49",     ["entre 40 a 44", "entre 45 a 49"]),
    ("50-59",     ["entre 50 a 54", "entre 55 a 59"]),
    ("60-69",     ["entre 60 a 64", "entre 65 a 69"]),
    ("70 OU MAIS", ["entre 70 a 74", "entre 75 a 79", "acima de 79"]),
]

SAIDA.parent.mkdir(parents=True, exist_ok=True)
print(f"✅ Pasta de saída: {SAIDA.parent}")
print(f"   Faixas configuradas: {[f for f, _ in FAIXAS]}")

✅ Pasta de saída: /home/anna/tcc/Seleção e Transformação/Mamografias
   Faixas configuradas: ['0-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70 OU MAIS']


---
## 3. Funções de Normalização de Texto

Funções para padronizar nomes de municípios e colunas dos CSVs brutos.

| Função | Responsabilidade |
|---|---|
| `remover_acentos` | Remove diacríticos via NFD |
| `normalizar_municipio` | Strip + lowercase + remove caracteres especiais |
| `normalizar_coluna` | Igual a `normalizar_municipio`, mas garante retorno string (para nomes de colunas) |
| `extrair_ano_do_nome_arquivo` | Extrai o ano (19xx/20xx) do nome do arquivo |
| `extrair_codigo_municipio` | Extrai código IBGE e nome do município da primeira coluna do CSV bruto |

In [3]:
def remover_acentos(texto: str) -> str:
    """Remove os diacríticos (acentos, cedilha etc.) do texto."""
    normalizado = unicodedata.normalize("NFD", texto)
    caracteres = []
    for caractere in normalizado:
        if unicodedata.category(caractere) != "Mn":
            caracteres.append(caractere)
    return "".join(caracteres)


def normalizar_municipio(valor: object) -> object:
    """Minúsculo, sem acentos e sem caracteres não alfanuméricos; devolve o valor original se não for string."""
    if not isinstance(valor, str):
        return valor
    texto = remover_acentos(valor).lower().strip()
    texto = re.sub(r"[^0-9a-z]+", " ", texto)
    return " ".join(texto.split())


def normalizar_coluna(texto: str) -> str:
    """Normaliza o nome da coluna; devolve string vazia se o resultado não for string."""
    resultado = normalizar_municipio(texto)
    if not isinstance(resultado, str):
        return ""
    return resultado


def extrair_ano_do_nome_arquivo(caminho: Path) -> int:
    """Extrai o ano (19xx ou 20xx) do nome do arquivo; levanta ValueError se não encontrar."""
    correspondencia = re.search(r"(19|20)\d{2}", caminho.stem)
    if not correspondencia:
        raise ValueError(f"Ano não encontrado no nome do arquivo: {caminho.name}")
    return int(correspondencia.group(0))


def extrair_codigo_municipio(valor: object) -> tuple:
    """Extrai (código, município) de '1234567 Nome do Município'; (None, None) se não casar."""
    if not isinstance(valor, str):
        return None, None
    texto = valor.strip().strip("\ufeff")
    correspondencia = re.match(r"\s*(\d{1,7})\s+(.*)$", texto)
    if not correspondencia:
        return None, None
    return correspondencia.group(1), correspondencia.group(2)


print("✅ Funções de normalização definidas.")

✅ Funções de normalização definidas.


---
## 4. Função de Carregamento por Ano

`carregar_por_ano` lê um único arquivo CSV anual e:
1. Extrai código IBGE e nome do município da primeira coluna
2. Agrupa as colunas brutas nas 8 faixas etárias padronizadas
3. Agrega por município (soma) e renomeia as colunas para `YYYY_faixa`

In [4]:
def carregar_por_ano(arquivo: Path) -> pd.DataFrame:
    """Lê o CSV anual de mamografias e devolve as colunas codigo, municipios e AAAA_faixa."""
    ano = extrair_ano_do_nome_arquivo(arquivo)
    tabela = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype=str)
    tabela.columns = tabela.columns.str.strip()

    col_municipio = tabela.columns[0]
    extraido = tabela[col_municipio].map(extrair_codigo_municipio)
    tabela[["codigo", "municipio_raw"]] = pd.DataFrame(extraido.tolist(), index=tabela.index)
    tabela = tabela.dropna(subset=["codigo", "municipio_raw"]).copy()

    tabela["codigo"] = tabela["codigo"].astype(str).str.strip().str.zfill(7)
    tabela["municipios"] = tabela["municipio_raw"].map(normalizar_municipio)
    tabela = tabela.drop(columns=["municipio_raw"])

    mapa_colunas = {}
    for coluna in tabela.columns:
        mapa_colunas[normalizar_coluna(coluna)] = coluna

    saida = tabela[["codigo", "municipios"]].copy()

    # Agrupa colunas brutas nas faixas padronizadas
    for faixa, padroes in FAIXAS:
        colunas_origem_encontradas = set()
        for padrao in padroes:
            for chave_norm, col_original in mapa_colunas.items():
                if padrao in chave_norm:
                    colunas_origem_encontradas.add(col_original)
        colunas_origem = sorted(colunas_origem_encontradas)

        if not colunas_origem:
            # Faixa não existe neste arquivo/ano: sem informação, não é zero.
            saida[faixa] = pd.Series(pd.NA, index=saida.index, dtype="Int64")
            continue

        saida[faixa] = (
            tabela[colunas_origem]
            .apply(lambda serie: pd.to_numeric(serie, errors="coerce"), axis=0)
            .sum(axis=1, min_count=1)
            .astype("Int64")
        )

    # Agrega por município; min_count=1 preserva NaN quando todas as subdivisões não têm informação
    colunas_faixas = [faixa for faixa, _ in FAIXAS]
    saida = (
        saida.groupby(["codigo", "municipios"], as_index=False)[colunas_faixas]
        .sum(min_count=1)
        .reset_index(drop=True)
    )

    renomeacao = {}
    for faixa in colunas_faixas:
        renomeacao[faixa] = f"{ano}_{faixa}"
    return saida.rename(columns=renomeacao)


print("✅ carregar_por_ano definida.")

✅ carregar_por_ano definida.


---
## 5. Pipeline de Tratamento

Executa o pipeline completo em 4 etapas:
1. Localiza todos os arquivos `mamografia_YYYY.csv`
2. Carrega e une ano a ano via `outer merge` pelo código do município
3. Calcula as colunas de total por ano (`YYYY_TOTAL`), agrupadas ao final da tabela
4. Enriquece com código IBGE completo (7 dígitos) e `REGIONAL DE SAÚDE` via tabela de municípios

In [5]:
# Etapa 1: Localizar arquivos
print("🔍 Localizando arquivos brutos...")
arquivos = sorted(ENTRADA.glob("mamografia_*.csv"))
if not arquivos:
    raise FileNotFoundError(f"Nenhum arquivo encontrado em: {ENTRADA}")

anos_encontrados = [extrair_ano_do_nome_arquivo(a) for a in arquivos]
print(f"   {len(arquivos)} arquivo(s): {anos_encontrados}")

🔍 Localizando arquivos brutos...
   13 arquivo(s): [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [6]:
# Etapa 2: Carregar e unir ano a ano
print("\n⚙️  Carregando e unindo anos...")
tabela: pd.DataFrame | None = None

for arquivo in arquivos:
    df_ano = carregar_por_ano(arquivo)
    ano = extrair_ano_do_nome_arquivo(arquivo)

    if tabela is None:
        tabela = df_ano
        print(f"   {ano}: {len(df_ano)} municípios (base)")
        continue

    tabela = tabela.merge(df_ano, on=["codigo"], how="outer", suffixes=("", "_dup"))

    # O outer join pode duplicar a coluna de município
    if "municipios_dup" in tabela.columns:
        tabela["municipios"] = tabela["municipios"].fillna(tabela["municipios_dup"])
        tabela = tabela.drop(columns=["municipios_dup"])

    print(f"   {ano}: {len(df_ano)} municípios → tabela acumulada: {len(tabela)}")

assert tabela is not None, "Nenhum arquivo foi processado."

# Ordena as colunas: municipios, codigo, depois AAAA_faixa por ano e faixa
ordem_faixas = [faixa for faixa, _ in FAIXAS]
anos_encontrados = set()
for coluna in tabela.columns:
    if re.match(r"^(19|20)\d{2}_", str(coluna)):
        anos_encontrados.add(int(coluna.split("_")[0]))
anos = sorted(anos_encontrados)

colunas_ano = []
for ano in anos:
    for faixa in ordem_faixas:
        if f"{ano}_{faixa}" in tabela.columns:
            colunas_ano.append(f"{ano}_{faixa}")
tabela = tabela[["municipios", "codigo"] + colunas_ano]

# Int64 anulável: município ausente no ano permanece nulo, não zero
tabela[colunas_ano] = (
    tabela[colunas_ano]
    .apply(lambda serie: pd.to_numeric(serie, errors="coerce"), axis=0)
    .astype("Int64")
)

print(f"\n✅ Tabela base: {tabela.shape}")


⚙️  Carregando e unindo anos...
   2013: 4 municípios (base)
   2014: 521 municípios → tabela acumulada: 521
   2015: 409 municípios → tabela acumulada: 583
   2016: 778 municípios → tabela acumulada: 813
   2017: 812 municípios → tabela acumulada: 840
   2018: 814 municípios → tabela acumulada: 846
   2019: 802 municípios → tabela acumulada: 849
   2020: 775 municípios → tabela acumulada: 849


   2021: 792 municípios → tabela acumulada: 850
   2022: 784 municípios → tabela acumulada: 851
   2023: 767 municípios → tabela acumulada: 851
   2024: 758 municípios → tabela acumulada: 851
   2025: 768 municípios → tabela acumulada: 852

✅ Tabela base: (852, 106)


In [7]:
# Etapa 3: Totais por ano
print("⚙️  Calculando totais...")
totais = {}

for ano in anos:
    colunas_do_ano = []
    for faixa in ordem_faixas:
        if f"{ano}_{faixa}" in tabela.columns:
            colunas_do_ano.append(f"{ano}_{faixa}")
    if not colunas_do_ano:
        continue
    totais[f"total_{ano}"] = tabela[colunas_do_ano].sum(axis=1, min_count=1).astype("Int64")

tabela = pd.concat([tabela, pd.DataFrame(totais)], axis=1)

# Reordena: municipios, codigo, faixas de todos os anos e, ao final, os totais
colunas_ordenadas = ["municipios", "codigo"]
for ano in anos:
    for faixa in ordem_faixas:
        if f"{ano}_{faixa}" in tabela.columns:
            colunas_ordenadas.append(f"{ano}_{faixa}")
for ano in anos:
    if f"total_{ano}" in tabela.columns:
        colunas_ordenadas.append(f"total_{ano}")

tabela = tabela[colunas_ordenadas].sort_values("codigo").reset_index(drop=True)
print(f"✅ Totais calculados. Shape: {tabela.shape}")

⚙️  Calculando totais...
✅ Totais calculados. Shape: (852, 119)


In [8]:
# Etapa 4: Enriquecimento com código IBGE completo e Regional de Saúde
print("\n⚙️  Enriquecendo com código IBGE e Regional de Saúde...")

try:
    df_mun = pd.read_csv(MUNICIPIOS, dtype=str)

    if "MUNICÍPIO NORMALIZADO" not in df_mun.columns:
        print("⚠️  Coluna 'MUNICÍPIO NORMALIZADO' não encontrada na tabela de municípios. Etapa ignorada.")
    else:
        # "MUNICÍPIO NORMALIZADO" da tabela de municípios preserva hífen/apóstrofo
        # (ex.: "guarda-mor", "olhos-d'agua") e por isso não batia com os nomes desta
        # base, normalizados sem hífen/apóstrofo (ex.: "guarda mor", "olhos d agua").
        # Monta-se o mapa a partir de duas fontes de nome, ambas renormalizadas aqui com a
        # mesma função desta base: "MUNICÍPIO ORIGINAL" (grafia oficial do IBGE) e
        # "MUNICÍPIO NORMALIZADO" (grafia já corrigida manualmente para casar com
        # divergências pontuais do DATASUS, como "dona eusebia" e "sao thome das letras").
        # Como as duas fontes de nome usam grafias distintas para o mesmo município em
        # alguns casos, a chave final combina as duas, sem perder nenhuma correspondência.
        mapa_por_original = {
            normalizar_municipio(nome): codigo
            for nome, codigo in zip(df_mun["MUNICÍPIO ORIGINAL"], df_mun["CÓDIGO IBGE"])
        }
        mapa_por_normalizado = {
            normalizar_municipio(nome): codigo
            for nome, codigo in zip(df_mun["MUNICÍPIO NORMALIZADO"], df_mun["CÓDIGO IBGE"])
        }
        mapa_ibge = {**mapa_por_original, **mapa_por_normalizado}

        tabela["codigo_ibge_mapeado"] = tabela["municipios"].map(mapa_ibge)
        tabela["codigo"] = tabela["codigo_ibge_mapeado"].fillna(tabela["codigo"])

        # Códigos curtos (< 7 dígitos) são expandidos para o código IBGE completo
        codigos_ibge = df_mun["CÓDIGO IBGE"].astype(str)
        mascara_curtos = tabela["codigo"].notna() & (tabela["codigo"].str.len() < 7)

        def expandir_codigo(codigo: str) -> str:
            """Expande um código curto para 7 dígitos se houver correspondência única na tabela IBGE."""
            if not isinstance(codigo, str) or len(codigo) >= 7:
                return codigo
            correspondencias = codigos_ibge[codigos_ibge.str.endswith(codigo)]
            if len(correspondencias) == 1:
                return correspondencias.iloc[0]
            return codigo

        if mascara_curtos.any():
            tabela.loc[mascara_curtos, "codigo"] = tabela.loc[mascara_curtos, "codigo"].map(expandir_codigo)
            print(f"   Códigos curtos expandidos: {int(mascara_curtos.sum())}")

        tabela = tabela.drop(columns=["codigo_ibge_mapeado"])

        sem_7_digitos = (tabela["codigo"].str.len() < 7).sum()
        print(f"   Municípios com código < 7 dígitos após expansão: {sem_7_digitos}")

        # Regional de Saúde pelo código IBGE completo
        mapa_regional = df_mun.set_index("CÓDIGO IBGE")["REGIONAL NORMALIZADA"].to_dict()
        tabela["regional"] = tabela["codigo"].map(mapa_regional)
        sem_regional = int(tabela["regional"].isna().sum())
        if sem_regional > 0:
            print(f"   ⚠️ {sem_regional} município(s) sem Regional de Saúde correspondente.")

        print(f"✅ Enriquecimento concluído.")

except FileNotFoundError:
    print(f"⚠️  Arquivo de municípios não encontrado: {MUNICIPIOS}. Código IBGE não foi enriquecido.")

# Nomes finais das colunas: MUNICÍPIO, CÓDIGO IBGE, REGIONAL DE SAÚDE, AAAA_TOTAL
renomeacao_final = {"municipios": "MUNICÍPIO", "codigo": "CÓDIGO IBGE", "regional": "REGIONAL DE SAÚDE"}
for ano in anos:
    renomeacao_final[f"total_{ano}"] = f"{ano}_TOTAL"
tabela = tabela.rename(columns=renomeacao_final)

# Reordena: MUNICÍPIO, CÓDIGO IBGE, REGIONAL DE SAÚDE, restante
colunas = tabela.columns.tolist()
colunas.remove("REGIONAL DE SAÚDE")
posicao = colunas.index("CÓDIGO IBGE") + 1
colunas.insert(posicao, "REGIONAL DE SAÚDE")
tabela = tabela[colunas]

print(f"\n✅ Tabela final: {tabela.shape}")


⚙️  Enriquecendo com código IBGE e Regional de Saúde...
   Municípios com código < 7 dígitos após expansão: 0
✅ Enriquecimento concluído.

✅ Tabela final: (852, 120)


---
## 6. Inspeção Pós-Processamento

Verificação da tabela final antes de salvar: shape, amostra de linhas, distribuição de totais e valores ausentes.

In [9]:
display(Markdown(f"**Shape final:** {tabela.shape[0]} linhas × {tabela.shape[1]} colunas"))

display(Markdown("**Primeiras linhas:**"))
colunas_total = []
for coluna in tabela.columns:
    if coluna.endswith("_TOTAL"):
        colunas_total.append(coluna)
colunas_previa = ["MUNICÍPIO", "CÓDIGO IBGE", "REGIONAL DE SAÚDE"] + colunas_total[:5]
display(tabela[colunas_previa].head(5))

# Estatísticas do total por município
total_por_municipio = tabela[colunas_total].sum(axis=1)
total_geral = int(total_por_municipio.sum())
media = total_por_municipio.mean()
mediana = total_por_municipio.median()
desvio = total_por_municipio.std()

print(f"\nTotal de exames no período : {total_geral:,}".replace(",", "."))
print(f"Média por município        : {media:,.0f}".replace(",", "."))
print(f"Mediana por município      : {mediana:,.0f}".replace(",", "."))
print(f"Desvio padrão              : {desvio:,.0f}".replace(",", "."))

ausentes = tabela.isnull().sum()
ausentes = ausentes[ausentes > 0]
if not ausentes.empty:
    display(Markdown(f"**Colunas com valores ausentes:** {len(ausentes)}"))
    display(ausentes.head(10).to_frame("Nulos").style.background_gradient(cmap="Reds"))
else:
    display(Markdown("✅ **Sem valores ausentes.**"))

**Shape final:** 852 linhas × 120 colunas

**Primeiras linhas:**

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,2013_TOTAL,2014_TOTAL,2015_TOTAL,2016_TOTAL,2017_TOTAL
0,abadia dos dourados,3100104,uberlandia,<NA>,155,84,84,445
1,abaete,3100203,sete lagoas,<NA>,<NA>,<NA>,218,31
2,abre campo,3100302,manhumirim,<NA>,2,231,301,382
3,acaiaca,3100401,ponte nova,<NA>,<NA>,<NA>,<NA>,<NA>
4,acucena,3100500,coronel fabriciano,<NA>,283,<NA>,375,85



Total de exames no período : 4.300.516
Média por município        : 5.048
Mediana por município      : 1.426
Desvio padrão              : 22.621


**Colunas com valores ausentes:** 117

,Nulos
2013_0-9,852
2013_10-19,852
2013_20-29,852
2013_30-39,848
2013_40-49,848
2013_50-59,848
2013_60-69,848
2013_70 OU MAIS,848
2014_0-9,331
2014_10-19,331


---
## 7. Salvando o Resultado

In [10]:
tabela.to_csv(SAIDA, index=False, encoding="utf-8-sig")

print("╔══════════════════════════════════════════════════════╗")
print("║              Arquivo gerado com sucesso              ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Arquivo : {str(SAIDA.name):<42}║")
print(f"║  Linhas  : {len(tabela):<42}║")
print(f"║  Colunas : {len(tabela.columns):<42}║")
print(f"║  Anos    : {str(anos):<42}║")
print("╚══════════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════════╗
║              Arquivo gerado com sucesso              ║
╠══════════════════════════════════════════════════════╣
║  Arquivo : tabela_mamografias_por_faixa_etária.csv   ║
║  Linhas  : 852                                       ║
║  Colunas : 120                                       ║
║  Anos    : [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]║
╚══════════════════════════════════════════════════════╝
